In [ ]:
%load_ext autoreload
%autoreload 2

# Run this cell first to prevent cached modules!

# RAG Ingestion Pipeline Verification

This notebook verifies the `RAGIngestion` class as required by the Stage 1 Prototyping rules.

In [ ]:
import sys
import os
sys.path.append("../../..")
from google.cloud import bigquery
from pipelines.enterprise_knowledge_base.rag_ingestion import RAGIngestion
from pipelines.enterprise_knowledge_base.orchestrator import KBIngestionPipeline

In [ ]:
# Initialize the pipeline orchestrator
PROJECT_ID = "ag-core-dev-fdx7"
os.environ["PROJECT_ID"] = PROJECT_ID
pipeline = KBIngestionPipeline(project_id=PROJECT_ID)


In [ ]:
# Run the full pipeline (staging + vectorization)
# Note: Replace with a real GCS URI available in your project
GCS_URI = "gs://kb-landing-zone-test/ingested/test_doc_for_embeddings.pdf"

try:
    pipeline.trigger_pipeline(GCS_URI)
except Exception as e:
    print(f'Pipeline execution result: {e}')

In [ ]:
# Verify BigQuery: Check if embeddings were generated
bq_client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT chunk_id, gcs_uri, ARRAY_LENGTH(embedding) as embedding_length 
FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
WHERE gcs_uri = '{GCS_URI}'
LIMIT 5
"""

results = bq_client.query(query).result()
for row in results:
     print(dict(row))